# Fine-tuning Chuẩn Y tế (v8 — Fix Double Sigmoid)
**Tác giả:** Me

### Changelog v8:
- Sửa lỗi `Double Sigmoid`: TorchXRayVision mặc định đã xuất ra Probabilities (chứa sẵn Sigmoid), nên dùng `BCELoss` thay vì `BCEWithLogitsLoss`.
- Bỏ `torch.sigmoid()` ở bước Validate/Test.
- Tự động tìm ngưỡng tối ưu (Optimal Threshold bằng Youden's J statistic) thay vì hardcode `0.5`.


## Bước 1: Kết nối Drive và Kaggle API

In [1]:
from google.colab import drive
import os, random
import numpy as np

# Seed toàn cục để đảm bảo reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

import torch
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

drive.mount('/content/drive')
model_save_path = '/content/drive/MyDrive/XAI_Models'
os.makedirs(model_save_path, exist_ok=True)

token = input('Kaggle Token (KGAT_...): ').strip()
if token.startswith('KGAT_'):
    os.environ['KAGGLE_API_TOKEN'] = token
    print('Xac thuc Kaggle thanh cong!')
else:
    raise ValueError('Token khong hop le!')


Mounted at /content/drive
Kaggle Token (KGAT_...): KGAT_aaea71a32c223c3bab4afbbec047c52c
Xac thuc Kaggle thanh cong!


## Bước 2: Tải + Trích xuất Chọn lọc + Split theo Patient ID

In [2]:
# Pin version de tranh API break
!pip install kaggle "torchxrayvision==1.4.0" scikit-image scikit-learn tqdm

import pandas as pd
import zipfile
from tqdm import tqdm
from sklearn.model_selection import train_test_split

!kaggle datasets download -d nih-chest-xrays/data -p /content/
print('Tai xong!')

zip_path = '/content/data.zip'
extract_folder = '/content/dataset'
os.makedirs(extract_folder, exist_ok=True)

with zipfile.ZipFile(zip_path, 'r') as z:
    csv_file = [f for f in z.namelist() if 'Data_Entry_2017' in f][0]
    z.extract(csv_file, '/content/')
    df = pd.read_csv(f'/content/{csv_file}')

    # Trich xuat Patient ID
    df['PatientID'] = df['Image Index'].str.split('_').str[0]

    df_eff  = df[df['Finding Labels'].str.contains('Effusion')].copy()
    df_eff['Label'] = 1.0
    df_norm = df[df['Finding Labels'] == 'No Finding'].copy()
    df_norm['Label'] = 0.0

    N = 3000
    df_pool = pd.concat([
        df_eff.sample(min(N, len(df_eff)),   random_state=SEED),
        df_norm.sample(min(N, len(df_norm)), random_state=SEED)
    ]).sample(frac=1, random_state=SEED).reset_index(drop=True)

    # Split theo Patient ID (tranh Data Leakage)
    all_pids = df_pool['PatientID'].unique()
    train_pids, temp_pids = train_test_split(all_pids, test_size=0.20, random_state=SEED)
    val_pids,  test_pids  = train_test_split(temp_pids, test_size=0.50, random_state=SEED)

    # FIX: Dung .copy() de tranh SettingWithCopyWarning
    df_train = df_pool[df_pool['PatientID'].isin(train_pids)].copy()
    df_val   = df_pool[df_pool['PatientID'].isin(val_pids)].copy()
    df_test  = df_pool[df_pool['PatientID'].isin(test_pids)].copy()

    print(f'\nSo luong mau: Train={len(df_train)} | Val={len(df_val)} | Test={len(df_test)}')

    # FIX: Kiem tra Class Imbalance sau khi split
    print('\n--- Phan bo nhan sau Patient-level Split ---')
    for name, d in [('Train', df_train), ('Val', df_val), ('Test', df_test)]:
        vc = d['Label'].value_counts()
        n_eff  = vc.get(1.0, 0)
        n_norm = vc.get(0.0, 0)
        ratio  = n_eff / max(n_norm, 1)
        print(f'  {name}: Effusion={n_eff} | Normal={n_norm} | Ratio={ratio:.2f}')
        if ratio < 0.5 or ratio > 2.0:
            print(f'  ⚠️ CANH BAO: {name} set mat can bang nghiem trong!')

    # Giai nen chi nhung anh can thiet
    all_needed = set(df_pool['Image Index'].tolist())
    extracted = {}
    for fi in tqdm(z.infolist(), desc='Trich xuat anh'):
        fn = os.path.basename(fi.filename)
        if fn in all_needed:
            z.extract(fi, extract_folder)
            extracted[fn] = os.path.join(extract_folder, fi.filename)

# FIX v7: Gan truc tiep tung bien, khong qua for-loop - het SettingWithCopyWarning
df_train['Path'] = df_train['Image Index'].map(extracted.get)
df_val['Path']   = df_val['Image Index'].map(extracted.get)
df_test['Path']  = df_test['Image Index'].map(extracted.get)

df_train = df_train.dropna(subset=['Path']).copy()
df_val   = df_val.dropna(subset=['Path']).copy()
df_test  = df_test.dropna(subset=['Path']).copy()

os.remove(zip_path)
print('\nDa xoa ZIP. San sang huan luyen!')


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 29.0/29.0 MB 38.9 MB/s eta 0:00:00
Dataset URL: https://www.kaggle.com/datasets/nih-chest-xrays/data
License(s): CC0-1.0
100% 42.0G/42.0G [18:54<00:00, 39.7MB/s]

Tai xong!

So luong mau: Train=4799 | Val=613 | Test=588

--- Phan bo nhan sau Patient-level Split ---
  Train: Effusion=2398 | Normal=2401 | Ratio=1.00
  Val: Effusion=313 | Normal=300 | Ratio=1.04
  Test: Effusion=289 | Normal=299 | Ratio=0.97


Trich xuat anh: 100%|██████████| 112128/112128 [00:34<00:00, 3236.72it/s]



Da xoa ZIP. San sang huan luyen!


## Bước 3: Dataset + Augmentation + worker_init_fn (Reproducible)

In [3]:
import torchvision.transforms as T
import torchxrayvision as xrv
from torch.utils.data import Dataset, DataLoader
from PIL import Image

class NIHDataset(Dataset):
    def __init__(self, df, augment=False):
        self.df = df.reset_index(drop=True)
        # Augmentation hop ly cho X-quang: flip ngang + xoay nhe
        # KHONG dung Color Jitter vi X-quang la grayscale y khoa
        self.aug = T.Compose([
            T.RandomHorizontalFlip(p=0.5),
            T.RandomRotation(degrees=5),
        ]) if augment else None

    def __len__(self):
        return len(self.df)

    def __getitem__(self, i):
        img = Image.open(self.df.iloc[i]['Path']).convert('L').resize((224, 224))
        if self.aug:
            img = self.aug(img)
        arr = xrv.datasets.normalize(np.array(img, dtype=np.float32), 255)[None]
        lbl = float(self.df.iloc[i]['Label'])
        return torch.from_numpy(arr), torch.tensor([lbl], dtype=torch.float32)

# FIX: worker_init_fn de augmentation reproducible voi num_workers > 0
def seed_worker(worker_id):
    np.random.seed(SEED + worker_id)
    random.seed(SEED + worker_id)

g = torch.Generator()
g.manual_seed(SEED)

train_dl = DataLoader(NIHDataset(df_train, augment=True),
                      batch_size=32, shuffle=True,  num_workers=2,
                      worker_init_fn=seed_worker, generator=g)
val_dl   = DataLoader(NIHDataset(df_val,   augment=False),
                      batch_size=32, shuffle=False, num_workers=2,
                      worker_init_fn=seed_worker)
test_dl  = DataLoader(NIHDataset(df_test,  augment=False),
                      batch_size=32, shuffle=False, num_workers=2,
                      worker_init_fn=seed_worker)

print(f'Train: {len(train_dl)} batches | Val: {len(val_dl)} | Test: {len(test_dl)}')


Train: 150 batches | Val: 20 | Test: 19


## Bước 4: Fine-tuning với LR=1e-5, AUC-ROC, Early Stopping

In [4]:
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import roc_auc_score

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

model = xrv.models.DenseNet(weights='densenet121-res224-all')

# Freeze tat ca layers, chi mo classifier
for param in model.parameters():
    param.requires_grad = False
for param in model.classifier.parameters():
    param.requires_grad = True

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f'Trainable: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)')

model.to(device)
effusion_idx = model.pathologies.index('Effusion')
print(f'Effusion index: {effusion_idx}')

# LR=1e-5 nhu comment (khong phai 1e-4)
optimizer = optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-5
)
# Scheduler theo AUC - bo verbose=True vi deprecated o PyTorch 2.x
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    # Bo verbose=True (deprecated PyTorch 2.x) - LR da in thu cong qua lr_now
    optimizer, mode='max', patience=2, factor=0.5
)
# Fix v8: XRV model da co apply_sigmoid=True mac dinh
# -> Output ra Probabilities -> Dung BCELoss thay vi BCEWithLogitsLoss
criterion = nn.BCELoss()

best_auc = 0.0
patience_counter = 0
PATIENCE = 4
EPOCHS   = 20
best_model_path = os.path.join(model_save_path, 'pleural_effusion_model_best_v6.pth')

print('\nBat dau huan luyen...\n')
for epoch in range(EPOCHS):
    # Train
    model.train()
    train_loss = 0.0
    for imgs, lbls in train_dl:
        imgs, lbls = imgs.to(device), lbls.to(device)
        optimizer.zero_grad()
        out  = model(imgs)[:, effusion_idx].unsqueeze(1)
        loss = criterion(out, lbls)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()

    # Validate
    model.eval()
    val_loss = 0.0
    all_probs, all_labels = [], []
    with torch.no_grad():
        for imgs, lbls in val_dl:
            imgs, lbls = imgs.to(device), lbls.to(device)
            out      = model(imgs)[:, effusion_idx].unsqueeze(1)
            val_loss += criterion(out, lbls).item()
            # Fix v8: Khong can sigmoid vi output da la xac suat
            probs = out.cpu().numpy().flatten()
            all_probs.extend(probs.tolist())
            all_labels.extend(lbls.cpu().numpy().flatten().tolist())

    try:
        auc = roc_auc_score(all_labels, all_probs)
    except Exception:
        auc = 0.0

    tl     = train_loss / len(train_dl)
    vl     = val_loss   / len(val_dl)
    lr_now = optimizer.param_groups[0]['lr']
    print(f'[{epoch+1:02d}/{EPOCHS}] Train={tl:.4f} | Val={vl:.4f} | AUC={auc:.4f} | LR={lr_now:.1e}')

    scheduler.step(auc)

    if auc > best_auc:
        best_auc = auc
        patience_counter = 0
        torch.save(model.state_dict(), best_model_path)
        print(f'   CHECKPOINT! AUC moi: {auc:.4f} -> Luu model!')
    else:
        patience_counter += 1
        print(f'   No improvement ({patience_counter}/{PATIENCE})')
        if patience_counter >= PATIENCE:
            print('Early Stopping! Model da dat toi uu.')
            break

print(f'\nBest Val AUC: {best_auc:.4f}')


Device: cuda
If this fails you can run `wget https://github.com/mlmed/torchxrayvision/releases/download/v1/nih-pc-chex-mimic_ch-google-openi-kaggle-densenet121-d121-tw-lr001-rot45-tr15-sc15-seed0-best.pt -O /root/.torchxrayvision/models_data/nih-pc-chex-mimic_ch-google-openi-kaggle-densenet121-d121-tw-lr001-rot45-tr15-sc15-seed0-best.pt`
[██████████████████████████████████████████████████]
Trainable: 18,450 / 6,966,034 (0.26%)
Effusion index: 7

Bat dau huan luyen...

[01/20] Train=0.5499 | Val=0.5208 | AUC=0.8802 | LR=1.0e-05
   CHECKPOINT! AUC moi: 0.8802 -> Luu model!
[02/20] Train=0.5524 | Val=0.5363 | AUC=0.8787 | LR=1.0e-05
   No improvement (1/4)
[03/20] Train=0.5426 | Val=0.5352 | AUC=0.8796 | LR=1.0e-05
   No improvement (2/4)
[04/20] Train=0.5484 | Val=0.5324 | AUC=0.8785 | LR=1.0e-05
   No improvement (3/4)
[05/20] Train=0.5218 | Val=0.4972 | AUC=0.8800 | LR=5.0e-06
   No improvement (4/4)
Early Stopping! Model da dat toi uu.

Best Val AUC: 0.8802


## Bước 5: Đánh giá cuối trên Test Set (AUC + Classification Report)

In [5]:
from sklearn.metrics import roc_auc_score, classification_report

# Load model tot nhat de danh gia khach quan
model.load_state_dict(torch.load(best_model_path, map_location=device))
model.eval()

all_probs, all_labels = [], []
with torch.no_grad():
    for imgs, lbls in test_dl:
        imgs = imgs.to(device)
        out  = model(imgs)[:, effusion_idx].unsqueeze(1)
        probs = out.cpu().numpy().flatten()
        all_probs.extend(probs.tolist())
        all_labels.extend(lbls.numpy().flatten().tolist())

# Fix v8: Tim nguong toi uu (Optimal Threshold) bang Youden's J statistic
from sklearn.metrics import roc_curve
fpr, tpr, thresholds = roc_curve(all_labels, all_probs)
optimal_idx = np.argmax(tpr - fpr)
optimal_threshold = thresholds[optimal_idx]
print(f'\n⭐ Nguong toi uu (Optimal Threshold): {optimal_threshold:.4f}\n')

preds_binary = [1 if p >= optimal_threshold else 0 for p in all_probs]

test_auc = roc_auc_score(all_labels, all_probs)
test_acc = sum(p == l for p, l in zip(preds_binary, all_labels)) / len(all_labels)

# FIX: In Classification Report day du
print('=' * 50)
print('KET QUA TREN TEST SET DOC LAP')
print('=' * 50)
print(f'  AUC-ROC  : {test_auc:.4f}')
print(f'  Accuracy : {100 * test_acc:.2f}%')
print()
print(classification_report(
    all_labels, preds_binary,
    target_names=['Normal', 'Effusion'],
    digits=4
))
print('=' * 50)
print(f'File da luu tai: {best_model_path}')
print('Tai file nay ve may va dat vao thu muc d:/My/XAI/models/')



⭐ Nguong toi uu (Optimal Threshold): 0.0682

KET QUA TREN TEST SET DOC LAP
  AUC-ROC  : 0.8732
  Accuracy : 81.46%

              precision    recall  f1-score   support

      Normal     0.8770    0.7391    0.8022       299
    Effusion     0.7679    0.8927    0.8256       289

    accuracy                         0.8146       588
   macro avg     0.8224    0.8159    0.8139       588
weighted avg     0.8233    0.8146    0.8137       588

File da luu tai: /content/drive/MyDrive/XAI_Models/pleural_effusion_model_best_v6.pth
Tai file nay ve may va dat vao thu muc d:/My/XAI/models/
